#

# 📊 NOTEBOOK 1: COLETA E PREPARAÇÃO DE DADOS
**Análise de Sentimento em Decisões Judiciais - TJCE**

## 🔧 1. CONFIGURAÇÃO DO AMBIENTE

### 1.1 Instalação de Dependências
**Descrição**: Instalação das bibliotecas Python necessárias.

In [1]:
# Instalar bibliotecas necessárias
!pip install -r requirements.txt

Ignoring pywin32: markers 'sys_platform == "win32"' don't match your environment


### 1.2 Importação de Bibliotecas

In [2]:
import pandas as pd
import json
from collections import Counter
import requests
import csv

## 📥 2. COLETA DE DADOS VIA API CNJ *DATAJUD*


### 2.1 Configuração da API e Parâmetros de Busca

 **Descrição**: Configuração da requisição à API pública do CNJ. O código de assunto 12487 refere-se a "Fornecimento de medicamentos", encontrado no site oficial do CNJ (https://www.cnj.jus.br/sgt/consulta_publica_assuntos.php).

 Para encontrar o código no site do cnj:
   1. Procure por "Medicamentos"
   2. Selecione a 4ª opção: `Fornecimento de medicamentos`
   3. Verifique o código que aparece, `12487 Fornecimento de medicamentos`

In [3]:
url = "https://api-publica.datajud.cnj.jus.br/api_publica_tjpa/_search"
api_key = "APIKey cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="

payload = json.dumps({
    "size": 10000,
    "query": {"match": {"assuntos.codigo": "12487"}},  # Código: Fornecimento de medicamentos
    "sort": [{"dataAjuizamento": {"order": "desc"}}]
})

headers = {
    'Authorization': api_key,
    'Content-Type': 'application/json'
}

### 2.2 Requisição e Validação dos Dados
**Descrição**: Execução da requisição HTTP POST para a API e validação inicial do volume de dados retornados.

In [4]:
response = requests.request("POST", url, headers=headers, data=payload)
dados_dict = response.json()
print(f"Total de processos encontrados na API: {dados_dict['hits']['total']['value']}")
print(f"Processos retornados nesta consulta: {len(dados_dict['hits']['hits'])}")

Total de processos encontrados na API: 210
Processos retornados nesta consulta: 210


## 🔄 3. PROCESSAMENTO E ESTRUTURAÇÃO DOS DADOS

### 3.1 Extração de Campos Essenciais
**Descrição**: Extração dos campos essenciais de cada processo (número, grau de jurisdição, data de ajuizamento e movimentos processuais) e criação do DataFrame principal.

In [5]:
processos = []

for hit in dados_dict['hits']['hits']:
    processo = hit['_source']

    numero_processo = processo['numeroProcesso']
    grau = processo['grau']
    # Use .get() to safely access 'dataAjuizamento', returning None if not found
    data_ajuizamento = processo.get('dataAjuizamento')
    movimentos = processo.get('movimentos', [])

    processos.append([
        numero_processo,
        grau,
        data_ajuizamento,
        movimentos
    ])

df = pd.DataFrame(
    processos,
    columns=[
        'numero_processo',
        'grau',
        'data_ajuizamento',
        'movimentos'
    ]
)

print(f"Total de processos no DataFrame: {len(df)}")
df.sample(5)

Total de processos no DataFrame: 210


,numero_processo,grau,data_ajuizamento,movimentos
199,08297392220228140301,JE,2022-03-10T10:01:59.000Z,"[{'complementosTabelados': [{'codigo': 2, 'val..."
61,08252816620258140006,JE,20251029095436,"[{'complementosTabelados': [{'codigo': 2, 'val..."
94,08636987620258140301,JE,20250630162925,"[{'complementosTabelados': [{'codigo': 2, 'val..."
12,08009383120268140051,G1,20260117155231,"[{'complementosTabelados': [{'codigo': 2, 'val..."
208,08593055020218140301,JE,2021-10-07T11:37:50.000Z,"[{'complementosTabelados': [{'codigo': 2, 'val..."


## 🔍 4. IDENTIFICAÇÃO E ANÁLISE DE DECISÕES

### 4.1 Definição de Termos de Decisão
**Descrição**: Definição dos termos-chave que caracterizam decisões judiciais relevantes para a análise.

In [6]:
from collections import Counter

termos_decisao = [
    "Procedência",
    "Improcedência",
    "Improcedência do pedido e improcedência do pedido contraposto",
    "Procedência do pedido e procedência do pedido contraposto",
    "Deferido",
    "Indeferido",
]

### 4.2 Extração de Decisões dos Movimentos Processuais
**Descrição**: Busca sistemática nos movimentos processuais por termos de decisão.

**Importante**: Filtra apenas processos de primeira instância (grau G1) para garantir homogeneidade na análise.

In [7]:
decisoes_por_processo = []
tipos_decisao_contagem = []

for _, row in df.iterrows():
    numero = row['numero_processo']
    movimentos = row['movimentos']
    grau = row['grau']
    data_ajuizamento = row['data_ajuizamento']

    decisoes_encontradas = []

    if movimentos:
        for mov in movimentos:
            nome_mov = mov.get('nome', '')

            # Filtrar apenas decisões de primeira instância (G1)
            if any(termo in nome_mov for termo in termos_decisao) and grau == 'G1':
                decisoes_encontradas.append(nome_mov)
                tipos_decisao_contagem.append(nome_mov)

    if decisoes_encontradas:
        decisoes_por_processo.append({
            'numero_processo': numero,
            'data_ajuizamento': data_ajuizamento,
            'decisoes': decisoes_encontradas,
            'grau': grau
        })

print(f"Processos com decisões: {len(decisoes_por_processo)} de {len(df)}")
print("\nTipos de decisões encontradas:")
for tipo, count in Counter(tipos_decisao_contagem).most_common(10):
    print(f"  {tipo}: {count}")

Processos com decisões: 15 de 210

Tipos de decisões encontradas:
  Procedência: 13
  Procedência em Parte: 1
  Improcedência: 1


### 4.3 Criação de DataFrame de Decisões
**Descrição**: Criação de um DataFrame específico para decisões, removendo duplicatas por processo (mantém apenas a primeira decisão encontrada).

In [8]:
decisoes_lista = []

for item in decisoes_por_processo:
    for decisao in item['decisoes']:
        decisoes_lista.append({
            'numero_processo': item['numero_processo'],
            'tipo_decisao': decisao,
            'grau': item['grau'],
            'data_ajuizamento': item['data_ajuizamento']
        })

df_decisoes = pd.DataFrame(decisoes_lista)
df_decisoes = df_decisoes.drop_duplicates(subset='numero_processo', keep='first')
df_decisoes.head(10)

,numero_processo,tipo_decisao,grau,data_ajuizamento
0,08045364520258140045,Procedência,G1,20250606130933
1,08019245120258140008,Procedência,G1,20250519143643
2,08007873120258140009,Procedência,G1,20250228104932
3,08020213020258140015,Procedência em Parte,G1,20250221141808
4,08001464120258140042,Procedência,G1,20250219181202
5,08000172120258140047,Procedência,G1,20250108184735
6,08021498820248140046,Procedência,G1,20241121130503
7,08031649420248140013,Procedência,G1,20240921110051
8,08070385420248140024,Procedência,G1,20240920121226
9,08012058620248140046,Procedência,G1,20240708160924


## 📊 5. SEGMENTAÇÃO POR TIPO DE DECISÃO

### 5.1 Separação por Categorias de Decisão
**Descrição**: Segmentação do dataset em categorias de decisão para análise comparativa posterior.

In [9]:
# Separar decisões por tipo
df_procedencia = df_decisoes[df_decisoes['tipo_decisao'] == 'Procedência'].copy()
df_improcedencia = df_decisoes[df_decisoes['tipo_decisao'] == 'Improcedência'].copy()

df_decisoes_completo = pd.concat([
    df_procedencia,
    df_improcedencia,
])

print(f"\n=== TODOS OS REGISTROS COM DECISÃO ===")
print(f"Total de registros: {len(df_decisoes_completo)}")
print(f"\nDistribuição por tipo:")
print(f"  - Procedência: {len(df_procedencia)}")
print(f"  - Improcedência: {len(df_improcedencia)}")

df = df_decisoes_completo

df


=== TODOS OS REGISTROS COM DECISÃO ===
Total de registros: 14

Distribuição por tipo:
  - Procedência: 13
  - Improcedência: 1


,numero_processo,tipo_decisao,grau,data_ajuizamento
0,08045364520258140045,Procedência,G1,20250606130933
1,08019245120258140008,Procedência,G1,20250519143643
2,08007873120258140009,Procedência,G1,20250228104932
4,08001464120258140042,Procedência,G1,20250219181202
5,08000172120258140047,Procedência,G1,20250108184735
6,08021498820248140046,Procedência,G1,20241121130503
7,08031649420248140013,Procedência,G1,20240921110051
8,08070385420248140024,Procedência,G1,20240920121226
9,08012058620248140046,Procedência,G1,20240708160924
11,08009505920228140027,Procedência,G1,20221104135458


## 🧹 6. LIMPEZA E CONSOLIDAÇÃO DO DATASET

### 6.1 Remoção de Valores Nulos

In [10]:
# Removendo registros NaN
df = df.dropna()
display(df.head(4))

,numero_processo,tipo_decisao,grau,data_ajuizamento
0,08045364520258140045,Procedência,G1,20250606130933
1,08019245120258140008,Procedência,G1,20250519143643
2,08007873120258140009,Procedência,G1,20250228104932
4,08001464120258140042,Procedência,G1,20250219181202


### 6.2 Reset de Índice

In [11]:
# Resetando index para manter organização do dataframe
df = df.reset_index(drop=True)
df

,numero_processo,tipo_decisao,grau,data_ajuizamento
0,08045364520258140045,Procedência,G1,20250606130933
1,08019245120258140008,Procedência,G1,20250519143643
2,08007873120258140009,Procedência,G1,20250228104932
3,08001464120258140042,Procedência,G1,20250219181202
4,08000172120258140047,Procedência,G1,20250108184735
5,08021498820248140046,Procedência,G1,20241121130503
6,08031649420248140013,Procedência,G1,20240921110051
7,08070385420248140024,Procedência,G1,20240920121226
8,08012058620248140046,Procedência,G1,20240708160924
9,08009505920228140027,Procedência,G1,20221104135458


## 📅 7. EXTRAÇÃO DE FEATURES TEMPORAIS

### 7.1 Extração de Dia da Semana e Ano
**Descrição**: Criação de features temporais (dia da semana e ano) para análise de padrões temporais nas decisões judiciais.

In [12]:
import pandas as pd

# Converter data_ajuizamento para datetime
df['data_ajuizamento_dt'] = pd.to_datetime(
    df['data_ajuizamento'],
    format='mixed', # Alterado para 'mixed' para lidar com diferentes formatos de data
    errors='coerce' # Adicionado para converter erros de parsing para NaT
)

# Remover linhas onde a conversão da data falhou, resultando em NaT
df.dropna(subset=['data_ajuizamento_dt'], inplace=True)

# Garantir que a coluna 'data_ajuizamento_dt' tenha o tipo de dado datetime64[ns]
df['data_ajuizamento_dt'] = pd.to_datetime(df['data_ajuizamento_dt'], utc=True)

dias_semana = {0: "Segunda", 1: "Terça", 2: "Quarta", 3: "Quinta", 4: "Sexta"}

# Extrair dia da semana (segunda = 0 | terça = 1 | ... | sexta = 4)
df['dia_semana'] = df['data_ajuizamento_dt'].dt.weekday.map(dias_semana)
df['ano'] = df['data_ajuizamento_dt'].dt.year

# Excluir coluna intermediária
df = df.drop(columns=['data_ajuizamento', 'data_ajuizamento_dt'])

df

/tmp/ipykernel_1788/838281323.py:4: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['data_ajuizamento_dt'] = pd.to_datetime(


,numero_processo,tipo_decisao,grau,dia_semana,ano
0,08045364520258140045,Procedência,G1,Sexta,2025
1,08019245120258140008,Procedência,G1,Segunda,2025
2,08007873120258140009,Procedência,G1,Sexta,2025
3,08001464120258140042,Procedência,G1,Quarta,2025
4,08000172120258140047,Procedência,G1,Quarta,2025
5,08021498820248140046,Procedência,G1,Quinta,2024
6,08031649420248140013,Procedência,G1,NaN,2024
7,08070385420248140024,Procedência,G1,Sexta,2024
8,08012058620248140046,Procedência,G1,Segunda,2024
9,08009505920228140027,Procedência,G1,Sexta,2022


### 7.2 Verificando ano mínimo e máximo do conjunto de dados

In [13]:
min = df['ano'].min()
max = df['ano'].max()

print(f"Ano mínimo: {min}")
print(f"Ano máximo: {max}")

Ano mínimo: 2022
Ano máximo: 2025


## 💾 8. EXPORTAÇÃO DE DADOS INTERMEDIÁRIOS

### 8.1 Exportação para CSV
**Descrição**: Exportação dos dados processados para uso posterior:

*  processos_decisao_dia_ano.csv: Dataset completo com decisões e features temporais
*  numeros_processos.csv: Lista única de números de processo para scraping de PDFs




In [14]:
df.to_csv("processos_dia_ano.csv", sep=';', index=False)
df['numero_processo'].drop_duplicates().to_csv("numeros_processos.csv", index=False)

print("\n=== ARQUIVOS EXPORTADOS ===")
print("  - processos_dia_ano.csv")
print("  - numeros_processos.csv")


=== ARQUIVOS EXPORTADOS ===
  - processos_dia_ano.csv
  - numeros_processos.csv


## 📄 9. INSTRUÇÕES PARA COLETA DE DOCUMENTOS COMPLETOS

### 9.1 Scraping de PDFs (Executar Localmente)
**Pré-requisitos:**
1. Clone o repositório: `git clone https://github.com/GiovanniBrigido/trabalho-final-deep-learning.git`
2. Navegue até o diretório do projeto
3. Execute os seguintes comandos:

**Comandos:**
```bash
python -m venv venv
pip install -r requirements.txt
playwright install chromium
python ./scraper_pdf_tjce.py
```

O que o script faz:

*   Lê o arquivo numeros_processos.csv
*   Acessa o site do TJCE
*   Baixa os PDFs das sentenças completas
*   Salva os arquivos na pasta data/decisoes





### 9.2 Extração de Texto dos PDFs (Executar Localmente)
Após gerar os arquivos básicos acima, execute o script `scraper_pdf_tjce.py` para realizar a extração de PDF's.

**Execute:**
```bash
python ./extrair_decisoes.py
```
**O que o script faz**:

* Lê todos os PDFs baixados
* Extrai o texto completo de cada sentença com a bib PyPDF2
* Gera o arquivo decisoes_extraidas.csv com:
   * Número do processo
   * Texto completo da decisão
   * Metadados adicionais

**Arquivo gerado:**

*  decisoes_extraidas.csv
* **!! SERÁ USADO NO NOTEBOOK 2 !!**